# Politische Achsen-Analyse (Waller & Anderson Logik)

Dieses Notebook berechnet die politische Achse basierend auf der Methode von Waller & Anderson (2021) im Basisjahr 2016 und wendet dieses "Lineal" auf alle ausgerichteten Jahre an.

**Voraussetzung:** Der Master-Loop im Notebook `Find_Anchors.ipynb` muss bereits durchgelaufen sein, damit die `projected_..._FULL.txt` Dateien existieren.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import gensim
from gensim.models import KeyedVectors

# Pfad zum Original-Code der Autoren hinzufügen
sys.path.append(os.path.abspath("."))

# Original-Funktionen importieren
from dimen_generation import DimenGenerator, score_embedding

def load_as_df(path):
    """Hilfsfunktion zum Laden und NORMALISIEREN von Word2Vec-TXT als Pandas DF."""
    model = KeyedVectors.load_word2vec_format(path, binary=False, unicode_errors='ignore')
    vectors = pd.DataFrame(model.vectors, index=model.index_to_key)
    
    # KRITISCHER SCHRITT: Normalisierung (L2-Norm) exakt wie in dimen_generation.py (Zeile 17)
    # Dies garantiert, dass die Länge jedes Vektors 1 ist und das spätere Dot-Product 
    # mathematisch identisch mit der Cosine Similarity ist.
    vectors = vectors.divide(np.linalg.norm(vectors.values, axis=1), axis='rows')
    
    return vectors

## 1. Die politische Achse im Basisjahr 2016 definieren

Hier nutzen wir den `DimenGenerator`, um aus einem Seed-Paar (z.B. liberal vs. conservative) eine stabile Dimension aus 10 Paaren zu erzeugen.

seed_wahlkampf = [('hillaryclinton', 'The_Donald')]
dim_wahl = generator.generate_dimension_from_seeds(seed_wahlkampf)

seed_ideologie = [('Liberal', 'Conservative')]

In [2]:
print("Lade Basis-Modell 2016...")
df_16 = load_as_df("Data/Vektoren/vektoren_2016.txt")

# Generator initialisieren (berechnet intern Nachbarschaften für die Dimensions-Suche)
generator = DimenGenerator(df_16)

# Seed-Paare definieren (WICHTIG: Groß-/Kleinschreibung muss mit dem Index übereinstimmen!)
# Z.B. 'Liberal' und 'Conservative' oder 'liberal' und 'conservative'
seeds = [('hillaryclinton', 'The_Donald')]

print("Generiere politische Dimension basierend auf 2016...")
dim_politics = generator.generate_dimension_from_seeds(seeds)

print("\nDie 10 gefundenen Paare für die Achse:")
for left, right in zip(dim_politics['left_comms'], dim_politics['right_comms']):
    print(f"{left:<20} <---> {right:<20}")

Lade Basis-Modell 2016...
166180 valid directions, 166180 calculated.
Generiere politische Dimension basierend auf 2016...

Die 10 gefundenen Paare für die Achse:
hillaryclinton       <---> The_Donald          
KamalaHarris         <---> GoogleMyBusiness    
Impeach_Trump        <---> HillaryMeltdown     
BlueMidterm2018      <---> tulsi               
democrats            <---> POLITIC             
Political_Revolution <---> HillaryForPrison    
AdventureCommunist   <---> crushcrush          
Pirate101            <---> FridayThe13thGame   
DetroitBecomeHuman   <---> accountsharing      
GunsAreCool          <---> progun              


Nachdem die Seeds für die Achse selbsständig gesucht wurden sind nu ndie folgenden Seeds vorhanden. aus diesen wird Achse gebaut

In [3]:
clean_seeds = [
    # 1. Ideologie
    ('Liberal', 'Conservative'),
    ('progressive', 'conservatives'),

    # 2. Partei & Harte Abgrenzung
    ('Democrat', 'Republican'),
    ('Political_Revolution', 'ConservativesOnly'),  # <-- NEU: 100% eindeutig!

    # 3. Frage-und-Antwort-Communitys
    ('AskALiberal', 'askaconservative'),
    ('AskDemocrats', 'AskTrumpSupporters'),
    ('askhillarysupporters', 'AskThe_Donald'),

    # 4. Wahlkampf & Kandidaten 2016
    ('hillaryclinton', 'The_Donald'),
    ('SandersForPresident', 'HillaryForPrison'),

    # 5. Affektive Polarisierung / Empörung
    ('Impeach_Trump', 'HillaryMeltdown')
]
df_16 = load_as_df("Data/Vektoren/vektoren_2016.txt")
print("Generiere politische Dimension aus eigenen Seeds...")
generator = DimenGenerator(df_16)
dim_politics = generator.generate_dimension_from_seeds(clean_seeds)

# Zur Kontrolle ausdrucken:
print("\nDie verwendeten Paare für die Achse:")
for left, right in zip(dim_politics['left_comms'], dim_politics['right_comms']):
    print(f"{left:<20} <---> {right:<20}")

Generiere politische Dimension aus eigenen Seeds...
166180 valid directions, 166180 calculated.
Eigene Seeds erkannt (10 Paare). Überspringe automatische Suche.

Die verwendeten Paare für die Achse:
Liberal              <---> Conservative        
progressive          <---> conservatives       
Democrat             <---> Republican          
Political_Revolution <---> ConservativesOnly   
AskALiberal          <---> askaconservative    
AskDemocrats         <---> AskTrumpSupporters  
askhillarysupporters <---> AskThe_Donald       
hillaryclinton       <---> The_Donald          
SandersForPresident  <---> HillaryForPrison    
Impeach_Trump        <---> HillaryMeltdown     


## 2. Berechnung der Scores für alle ausgerichteten Jahre

Wir nutzen nun den 2016er Vektor (`dim_politics`), um alle Subreddits in allen Jahren auf dieses feste Lineal zu projizieren.

In [4]:
import os
import pandas as pd

JAHRE = ["2016", "2017","2018","2019","2020","2021","2022","2023","2024"]

# Wir nutzen eine Liste statt eines Dictionaries, das macht das Zusammenfügen leichter
all_scores_list = []

for jahr in JAHRE:
    if jahr == "2016":
        print("Berechne Scores für Jahr 2016 (aus Arbeitsspeicher)...")
        # Wir greifen direkt auf die bestehende Variable zu!
        df_current = df_16
    else:
        path = f"SeNSe-main/SeNSe-main/output/projected_{jahr[2:]}_onto_16_FULL.txt"
        if not os.path.exists(path):
            print(f"Überspringe {jahr}, Datei nicht gefunden: {path}")
            continue

        print(f"Berechne Scores für Jahr {jahr}...")
        df_current = load_as_df(path)

    # 1. Scores berechnen
    dimensions_to_score = [("partisan", dim_politics)]
    scores_df = score_embedding(df_current, dimensions_to_score)

    # 2. Die Spalte umbenennen, damit in der CSV nicht 9x "partisan" steht,
    # sondern "partisan_2016", "partisan_2017" etc.
    scores_df.columns = [f"partisan_{jahr}"]

    # 3. Den fertigen DataFrame des Jahres in unsere Liste packen
    all_scores_list.append(scores_df)

# 4. Alle DataFrames anhand der Subreddit-Namen (Index) nebeneinander mergen
df_scores_final = pd.concat(all_scores_list, axis=1)
mean_2016 = df_scores_final["partisan_2016"].mean()
sd_2016 = df_scores_final["partisan_2016"].std()

df_scores_z = (df_scores_final - mean_2016) / sd_2016
df_scores_z.columns = [c.replace("partisan_", "partisan_z_") for c in df_scores_z.columns]
#df_scores_z.to_csv("Data/Vektoren/all_scores_z.csv", index=True)

# 5. Speichern! index=True ist absolut essenziell, damit die Subreddit-Namen erhalten bleiben
speicher_pfad = "Data/Vektoren/all_scores.csv"
#df_scores_final.to_csv(speicher_pfad, index=True)

print(f"\nAlle Scores erfolgreich berechnet und unter {speicher_pfad} gespeichert!")

Berechne Scores für Jahr 2016 (aus Arbeitsspeicher)...
Berechne Scores für Jahr 2017...
Berechne Scores für Jahr 2018...
Berechne Scores für Jahr 2019...
Berechne Scores für Jahr 2020...
Berechne Scores für Jahr 2021...
Berechne Scores für Jahr 2022...
Berechne Scores für Jahr 2023...
Berechne Scores für Jahr 2024...

Alle Scores erfolgreich berechnet und unter Data/Vektoren/all_scores.csv gespeichert!


## 3. Analyse der zeitlichen Entwicklung

Hier führen wir die Daten zusammen, um die Bewegung eines Subreddits über die Jahre zu sehen.

In [5]:
# 1. Wir führen die DataFrames zusammen (Concat mit axis=1 und join='outer')
# 'outer' sorgt dafür, dass ALLE Subreddits, die in irgendeinem Jahr auftauchten, erhalten bleiben.
# Fehlende Werte (NaN) entstehen dort, wo ein Subreddit in einem Jahr noch nicht existierte.
df_scores_final = pd.concat(all_scores_list, axis=1, join='outer')

# 2. Jetzt erstellen wir die final_table, basierend auf dem Index von df_scores_final (dem "Super-Index")
final_table = pd.DataFrame(index=df_scores_final.index)

# 3. Spalten befüllen
for col in df_scores_final.columns:
    jahr = col.split("_")[1]
    final_table[f"score_{jahr}"] = df_scores_final[col]

# 4. CSV speichern
#final_table.to_csv("politischer_wandel_2016_2024.csv")

print(f"Tabelle erstellt mit {len(final_table)} Subreddits (inkl. später hinzugekommener).")

Tabelle erstellt mit 31284 Subreddits (inkl. später hinzugekommener).


In [6]:
"""
Diagnose-Zelle zur Delle 2023 (FF2).

GEDACHT ALS NEUE ZELLE am Ende von `Political_Axis_Analysis.ipynb`.
Sie setzt voraus, dass dort schon `load_as_df` (normalisiert die Vektoren zeilenweise
auf Laenge 1) und `clean_seeds` (die 10 Seed-Paare der politischen Achse) definiert
sind. Genau deshalb gehoert sie in dieses Notebook und nicht in ein Wegwerf-Skript:
sie rechnet mit exakt denselben Funktionen, aus denen `all_scores.csv` entstanden ist.

WARUM DIESE GROESSEN UND NICHT DIE AUS DEM HANDOFF
--------------------------------------------------
`HANDOFF_DELLE_2023.md` fragt nach Achsen-Norm, Procrustes-Skalierungsfaktor und
mittlerer Vektornorm pro Jahr. Diese drei Groessen koennen die Delle nicht erklaeren,
das ergibt schon der Code:

  1. Das Alignment ist ORTHOGONALES Procrustes ohne Skalenterm
     (`SeNSe-main/SeNSe-main/ALIGNMENT_MODERN.py`, Z. 26-37: L2-Normierung der Anker,
     dann reine SVD-Rotation). Einen Skalierungsfaktor gibt es nicht.
  2. `load_as_df` normiert JEDEN Vektor auf Laenge 1, und `score_embedding`
     (`dimen_generation.py`, Z. 27) normiert zusaetzlich die Achse. Der Partisan-Score
     ist also ein reiner Cosinus. Eine gleichmaessige Stauchung faellt heraus.
  3. Die z-Standardisierung nutzt Mittelwert und SD von 2016 fuer ALLE Jahre
     (ein globaler Faktor). Die rohe SD pro Jahr ist deshalb keine unabhaengige
     Kontrolle, sondern dieselbe Reihe mal einer Konstanten.

Bleibt als Mechanismus die ROTATIONSQUALITAET des jeweiligen Jahres. Eine orthogonale
Rotation laesst die Geometrie INNERHALB eines Jahres unveraendert (Abstaende und Winkel
bleiben, deshalb sieht Moran nichts), sie entscheidet aber darueber, wie gut die feste
2016er-Achse im Raum des Zieljahres noch die politische Richtung trifft. Sitzt die
2023er-Rotation schlechter, projizieren alle Subreddits flacher auf das Lineal, die
Score-Streuung sinkt in JEDER Teilpopulation gleichzeitig, und die Geometrie macht es
nicht mit. Das ist genau das beobachtete Muster.

WAS DIE ZELLE MISST
-------------------
  cos_achse_zu_2016 : Die politische Achse wird pro Jahr im AUSGERICHTETEN Raum des
                      Jahres aus denselben 10 Seed-Paaren neu gebaut und mit der
                      2016er-Achse verglichen. Waere das Alignment perfekt und die
                      politische Struktur stabil, laege der Wert nahe 1. Ein Einbruch
                      2023 ist der direkte Beleg.
  anker_n, anker_*  : Zahl und Guete der Ankerpaare, mit denen die Rotation des Jahres
                      geschaetzt wurde (aus `final_anker_16_{jahr}_full.csv`). Wenige
                      oder schlechte Anker sind die naheliegende Ursache einer
                      schlechteren Rotation.
  score_sd_roh      : Rohe SD des Partisan-Scores pro Jahr ueber alle Subreddits des
                      Jahres, nur zur Einordnung (siehe Punkt 3 oben, keine
                      unabhaengige Kontrolle).

Ergebnis geht nach `Auswertung_CSV/ff2_delle_alignment.csv`.
"""

import numpy as np
import pandas as pd
import os

JAHRE = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


def achse_aus_seeds(df, seeds):
    """Achse wie in dimen_generation.py bei >1 expliziten Seed-Paaren (Z. 79-86):
    Differenzvektoren rechts minus links, gemittelt, auf Laenge 1 normiert."""
    paare = [(l, r) for l, r in seeds if l in df.index and r in df.index]
    fehlend = [(l, r) for l, r in seeds if (l, r) not in paare]
    d = df.loc[[r for _, r in paare]].values - df.loc[[l for l, _ in paare]].values
    v = d.mean(axis=0)
    return v / np.linalg.norm(v), len(paare), fehlend


# --- 2016er-Achse als Referenz (Basisjahr, unausgerichtet = Zielraum) -------------
df_16_diag = load_as_df("Data/Vektoren/vektoren_2016.txt")
achse_16, n16, fehl16 = achse_aus_seeds(df_16_diag, clean_seeds)
print(f"2016: Achse aus {n16}/{len(clean_seeds)} Seed-Paaren gebaut")
if fehl16:
    print(f"  fehlende Paare 2016: {fehl16}")

scores = pd.read_csv("Data/Vektoren/all_scores.csv", index_col=0)

zeilen = []
for jahr in JAHRE:
    if jahr == 2016:
        df_j = df_16_diag
    else:
        pfad = f"SeNSe-main/SeNSe-main/output/projected_{str(jahr)[2:]}_onto_16_FULL.txt"
        if not os.path.exists(pfad):
            print(f"{jahr}: Datei fehlt, übersprungen ({pfad})")
            continue
        df_j = load_as_df(pfad)

    achse_j, n_paare, fehlend = achse_aus_seeds(df_j, clean_seeds)
    cos_j = float(np.dot(achse_j, achse_16))

    # Ankergüte der Rotation dieses Jahres
    ank_pfad = f"final_anker_16_{jahr}_full.csv"
    if jahr != 2016 and os.path.exists(ank_pfad):
        ank = pd.read_csv(ank_pfad)
        a_n, a_mean, a_med, a_min = len(ank), ank["score"].mean(), ank["score"].median(), ank["score"].min()
    else:
        a_n = a_mean = a_med = a_min = np.nan

    sp = f"partisan_{jahr}"
    sd_roh = scores[sp].std() if sp in scores.columns else np.nan

    zeilen.append(dict(jahr=jahr, seed_paare=n_paare, cos_achse_zu_2016=cos_j,
                       anker_n=a_n, anker_score_mean=a_mean, anker_score_median=a_med,
                       anker_score_min=a_min, score_sd_roh=sd_roh))
    print(f"{jahr}: cos(Achse, 2016) = {cos_j:.4f} | Seedpaare {n_paare} | "
          f"Anker {a_n if a_n == a_n else '-'} | SD_roh {sd_roh:.4f}")
    if fehlend:
        print(f"   Achtung, Seed-Paare in {jahr} nicht vorhanden: {fehlend}")

diag = pd.DataFrame(zeilen)
os.makedirs("Auswertung_CSV", exist_ok=True)
diag.to_csv("Auswertung_CSV/ff2_delle_alignment.csv", index=False)
print("\n-> Auswertung_CSV/ff2_delle_alignment.csv")
print(diag.to_string(index=False))

# --- Lesehilfe -------------------------------------------------------------------
if len(diag) > 2:
    d = diag.set_index("jahr")
    if 2023 in d.index:
        umfeld = [j for j in (2022, 2024) if j in d.index]
        if umfeld:
            ref = d.loc[umfeld, "cos_achse_zu_2016"].mean()
            delta = (d.loc[2023, "cos_achse_zu_2016"] - ref) / ref * 100
            print(f"\n2023 gegen den Mittelwert von {umfeld}: "
                  f"{delta:+.2f} % bei cos(Achse, 2016).")
            print("Deutlich negativ = Rotationsartefakt bestätigt. "
                  "Nahe null = Delle kommt NICHT aus dem Alignment, dann bleiben "
                  "Aktivitaet/Korpus (Aufgabe 2) als Erklaerung.")


2016: Achse aus 10/10 Seed-Paaren gebaut
2016: cos(Achse, 2016) = 1.0000 | Seedpaare 10 | Anker - | SD_roh 0.0675
2017: cos(Achse, 2016) = 0.3643 | Seedpaare 10 | Anker 2368 | SD_roh 0.0677
2018: cos(Achse, 2016) = 0.4074 | Seedpaare 9 | Anker 2346 | SD_roh 0.0679
   Achtung, Seed-Paare in 2018 nicht vorhanden: [('askhillarysupporters', 'AskThe_Donald')]
2019: cos(Achse, 2016) = 0.4325 | Seedpaare 9 | Anker 2327 | SD_roh 0.0683
   Achtung, Seed-Paare in 2019 nicht vorhanden: [('askhillarysupporters', 'AskThe_Donald')]
2020: cos(Achse, 2016) = 0.3370 | Seedpaare 9 | Anker 2346 | SD_roh 0.0680
   Achtung, Seed-Paare in 2020 nicht vorhanden: [('askhillarysupporters', 'AskThe_Donald')]
2021: cos(Achse, 2016) = 0.2850 | Seedpaare 8 | Anker 2313 | SD_roh 0.0679
   Achtung, Seed-Paare in 2021 nicht vorhanden: [('askhillarysupporters', 'AskThe_Donald'), ('hillaryclinton', 'The_Donald')]
2022: cos(Achse, 2016) = 0.2670 | Seedpaare 8 | Anker 2300 | SD_roh 0.0705
   Achtung, Seed-Paare in 2022 ni

In [7]:
"""
Placebo-Achsen-Test zur Delle 2023 (FF2).

GEDACHT ALS NEUE ZELLE am Ende von `Political_Axis_Analysis.ipynb`, direkt hinter
der Delle-Diagnose. Setzt `load_as_df` und `clean_seeds` aus diesem Notebook voraus.

FRAGE
-----
Die SD des rohen Partisan-Scores bricht 2023 um 2,7 Prozent ein (0,07051 ->
0,06860 -> 0,07006) und erholt sich 2024. Aus der Delle-Diagnose ist bereits raus:
Alignment-Guete, Ankerzahl, Populationsvolumen, Komposition, Auswertungsschritte.
Uebrig sind zwei Erklaerungen, die sich ausschliessen:

  1. GLOBAL. Der 2023er-Raum ist insgesamt weniger scharf, alle Projektionen
     schrumpfen Richtung null, egal auf welche Richtung. Dann ist die politische
     Achse nur ein Beispiel und die Delle ein Messartefakt.
  2. SPEZIFISCH. Nur entlang der politischen Achse ist 2023 gestaucht. Dann waere
     es ein inhaltlicher Befund.

TESTAUFBAU
----------
Pro Jahr werden R zufaellige Einheitsrichtungen im ausgerichteten 150-dim Raum
gezogen und die SD der Projektionen gemessen, also genau die Groesse, die auch
der Partisan-Score liefert (die Vektoren sind zeilenweise L2-normiert, das
Skalarprodukt IST der Cosinus).

WICHTIG: Es werden fuer ALLE Jahre DIESELBEN Zufallsrichtungen verwendet
(fester Seed, einmal gezogen). Sonst mischt sich Ziehungsrauschen in den
Jahresvergleich.

Die entscheidende Groesse ist nicht die Placebo-SD allein, sondern das
VERHAELTNIS sd_partisan / sd_placebo:
  - Verhaeltnis 2023 flach  -> die politische Achse schrumpft nur so weit wie
                               alles andere auch  -> Fall 1, globales Artefakt
  - Verhaeltnis 2023 bricht ein -> spezifisch politische Kompression -> Fall 2

ZWEI STICHPROBEN
----------------
  "jahr"  = alle Subreddits im Jahresraum (entspricht der Spalte score_sd_roh
            aus der Delle-Diagnose, aber die Zusammensetzung wechselt)
  "panel" = nur Subreddits, die in ALLEN neun Jahren vorkommen. Damit ist jede
            Kompositionswirkung ausgeschlossen und es bleibt die reine Geometrie.

Ergebnis geht nach `Auswertung_CSV/ff2_placebo_achse.csv`.
"""

import numpy as np
import pandas as pd
import os

JAHRE = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
R_RICHTUNGEN = 200
SEED = 20260723


def pfad_fuer(jahr):
    if jahr == 2016:
        return "Data/Vektoren/vektoren_2016.txt"
    return f"SeNSe-main/SeNSe-main/output/projected_{str(jahr)[2:]}_onto_16_FULL.txt"


def achse_aus_seeds(df, seeds):
    """Wie in dimen_generation.py bei mehreren expliziten Seed-Paaren:
    Differenzvektoren rechts minus links, gemittelt, auf Laenge 1 normiert."""
    paare = [(l, r) for l, r in seeds if l in df.index and r in df.index]
    d = df.loc[[r for _, r in paare]].values - df.loc[[l for l, _ in paare]].values
    v = d.mean(axis=0)
    return v / np.linalg.norm(v), len(paare)


# --- alle Jahre einmal laden ------------------------------------------------
print("Lade Jahresräume ...", flush=True)
raeume = {}
for j in JAHRE:
    p = pfad_fuer(j)
    if not os.path.exists(p):
        print(f"  {j}: Datei fehlt, übersprungen ({p})")
        continue
    raeume[j] = load_as_df(p)
    print(f"  {j}: {raeume[j].shape[0]:>6} Subs x {raeume[j].shape[1]} Dim", flush=True)

DIM = next(iter(raeume.values())).shape[1]

# Panel = in allen geladenen Jahren vorhanden
panel = set.intersection(*(set(df.index) for df in raeume.values()))
panel = sorted(panel)
print(f"\nPanel (in allen {len(raeume)} Jahren vorhanden): {len(panel)} Subreddits")

# --- feste politische Achse aus 2016 ----------------------------------------
if "dim_politics" in dir():
    achse_pol = np.asarray(dim_politics["vector"], dtype=float)
    achse_pol = achse_pol / np.linalg.norm(achse_pol)
    print("Politische Achse: aus dim_politics übernommen")
else:
    achse_pol, n_paare = achse_aus_seeds(raeume[2016], clean_seeds)
    print(f"Politische Achse: aus {n_paare} clean_seeds in 2016 neu gebaut")

# --- EINMAL gezogene Zufallsrichtungen, für alle Jahre dieselben -----------
rng = np.random.default_rng(SEED)
Q = rng.normal(size=(DIM, R_RICHTUNGEN))
Q = Q / np.linalg.norm(Q, axis=0, keepdims=True)
print(f"{R_RICHTUNGEN} Zufallsrichtungen gezogen (Seed {SEED})\n")

iso = 1.0 / np.sqrt(DIM)
print(f"Zur Einordnung: bei perfekt isotroper Wolke waere die SD ~ 1/sqrt({DIM}) = {iso:.4f}\n")

# --- messen -----------------------------------------------------------------
zeilen = []
for stichprobe in ("jahr", "panel"):
    for j, df in raeume.items():
        V = df.values if stichprobe == "jahr" else df.loc[panel].values

        sd_pol = float(np.dot(V, achse_pol).std(ddof=1))
        proj = V @ Q                       # (n_subs, R)
        sd_plac = proj.std(axis=0, ddof=1)  # eine SD je Zufallsrichtung

        zeilen.append(dict(
            stichprobe=stichprobe, jahr=j, n_subs=V.shape[0],
            sd_partisan=sd_pol,
            sd_placebo_mean=float(sd_plac.mean()),
            sd_placebo_median=float(np.median(sd_plac)),
            sd_placebo_p05=float(np.percentile(sd_plac, 5)),
            sd_placebo_p95=float(np.percentile(sd_plac, 95)),
            verhaeltnis=sd_pol / float(sd_plac.mean()),
        ))

erg = pd.DataFrame(zeilen)
os.makedirs("Auswertung_CSV", exist_ok=True)
erg.to_csv("Auswertung_CSV/ff2_placebo_achse.csv", index=False)

pd.set_option("display.width", 160)
for stichprobe in ("jahr", "panel"):
    print(f"--- Stichprobe: {stichprobe} ---")
    print(erg[erg.stichprobe == stichprobe][
        ["jahr", "n_subs", "sd_partisan", "sd_placebo_mean", "verhaeltnis"]
    ].to_string(index=False, float_format=lambda x: f"{x:.5f}"))
    print()

print("-> Auswertung_CSV/ff2_placebo_achse.csv\n")

# --- Lesehilfe: 2023 gegen das Mittel aus 2022 und 2024 ---------------------
print("2023 gegen den Mittelwert aus 2022 und 2024, in Prozent:\n")
print(f"{'Stichprobe':<12}{'sd_partisan':>14}{'sd_placebo':>14}{'Verhaeltnis':>14}")
for stichprobe in ("jahr", "panel"):
    d = erg[erg.stichprobe == stichprobe].set_index("jahr")
    if not {2022, 2023, 2024}.issubset(d.index):
        continue
    aus = [f"{stichprobe:<12}"]
    for spalte in ("sd_partisan", "sd_placebo_mean", "verhaeltnis"):
        ref = d.loc[[2022, 2024], spalte].mean()
        aus.append(f"{(d.loc[2023, spalte] - ref) / ref * 100:>+13.2f}%")
    print("".join(aus))

print("""
LESEREGEL
  sd_placebo faellt 2023 aehnlich stark wie sd_partisan, Verhaeltnis bleibt flach
      -> Fall 1: globale Kompression des Embeddings, nicht politikspezifisch.
         Fuer den Text ein Mechanismus, keine Etikettierung. Die URSACHE
         (Aktivitaet 2023) bleibt trotzdem offen und braucht Aufgabe 2.

  sd_placebo bleibt flach, Verhaeltnis bricht 2023 ein
      -> Fall 2: spezifisch entlang der politischen Achse gestaucht.
         Dann ist es kein Messartefakt und gehoert inhaltlich diskutiert.
""")


Lade Jahresraeume ...
  2016:  16618 Subs x 150 Dim
  2017:  18426 Subs x 150 Dim
  2018:  20255 Subs x 150 Dim
  2019:  22409 Subs x 150 Dim
  2020:  24842 Subs x 150 Dim
  2021:  26930 Subs x 150 Dim
  2022:  28640 Subs x 150 Dim
  2023:  29446 Subs x 150 Dim
  2024:  28848 Subs x 150 Dim

Panel (in allen 9 Jahren vorhanden): 15522 Subreddits
Politische Achse: aus dim_politics uebernommen
200 Zufallsrichtungen gezogen (Seed 20260723)

Zur Einordnung: bei perfekt isotroper Wolke waere die SD ~ 1/sqrt(150) = 0.0816

--- Stichprobe: jahr ---
 jahr  n_subs  sd_partisan  sd_placebo_mean  verhaeltnis
 2016   16618      0.06748          0.06986      0.96586
 2017   18426      0.06771          0.07005      0.96662
 2018   20255      0.06790          0.07038      0.96478
 2019   22409      0.06832          0.07048      0.96944
 2020   24842      0.06801          0.07041      0.96600
 2021   26930      0.06795          0.07016      0.96849
 2022   28640      0.07051          0.07049      1.000